In [1]:
import os
import traceback
import multiprocessing
import warnings
warnings.filterwarnings("ignore")

import pickle
import numpy as np
import common_value
from utils import get_merged_data_hdf5
from pig_oestrus_features import PigOestrusFeatureExtractor
from ml_utils import getfeature

In [2]:
# hdf5_file = r'./data' # 计算特征源文件路径

处理得到特征

In [3]:
# have_no_dataset, have_no_label, alive_dead_dataset, alive_dead_label, nolabel_feature, logic_label = getfeature(hdf5_file)


保存数据集和标签

In [ ]:
# import pickle
# dataset_file = r'./dataset'
# label_file = r'./label'
# if not os.path.exists(dataset_file):
#     os.makedirs(dataset_file)
# if not os.path.exists(label_file):
#     os.makedirs(label_file)

# with open(dataset_file+'/have_no_dataset.pkl', 'wb') as f:
#     pickle.dump(have_no_dataset, f)
# with open(dataset_file+'/alive_dead_dataset.pkl', 'wb') as f:
#     pickle.dump(alive_dead_dataset, f)
# with open(dataset_file+'/nolabel_feature.pkl', 'wb') as f:
#     pickle.dump(nolabel_feature, f)

# with open(label_file+'/have_no_label.txt', 'w') as f:
#     for label in have_no_label:
#         f.write(f'{label}\n')
# with open(label_file+'/alive_dead_label.txt', 'w') as f:
#     for label in alive_dead_label:
#         f.write(f'{label}\n')
# with open(label_file+'/logic_label.txt', 'w') as f:
#     for label in logic_label:
#         f.write(f'{label}\n')

读取数据集以及标签

In [2]:
have_no_dataset_file = r'./dataset/have_no_dataset.pkl'
alive_dead_dataset_file = r'./dataset/alive_dead_dataset.pkl'
nolabel_feature_file = r'./dataset/nolabel_feature.pkl'

have_no_label_file = r'./label/have_no_label.txt'
alive_dead_label_file = r'./label/alive_dead_label.txt'
logic_label_file = r'./label/logic_label.txt'

In [3]:
with open(have_no_dataset_file, 'rb') as f:
    have_no_dataset = pickle.load(f)
with open(alive_dead_dataset_file, 'rb') as f:
    alive_dead_dataset = pickle.load(f)
with open(nolabel_feature_file, 'rb') as f:
    nolabel_dataset = pickle.load(f)

In [4]:
have_no_label = []
alive_dead_label = []
logic_label = []
with open(have_no_label_file, 'r') as f:
    for line in f.readlines():
        have_no_label.append(int(line.strip()))
with open(alive_dead_label_file, 'r') as f:
    for line in f.readlines():
        alive_dead_label.append(int(line.strip()))
with open(logic_label_file, 'r') as f:
    for line in f.readlines():
        logic_label.append(int(line.strip()))

In [5]:
import pandas as pd

In [6]:
have_no_df = pd.DataFrame(have_no_dataset, columns=['amp_max', 'amp_min', 'amp_mean', 'max_channel', 'ideal_channel', 'br_amp', 'up_down'])
alive_dead_df = pd.DataFrame(alive_dead_dataset, columns=['amp_max', 'amp_min', 'amp_mean', 'max_channel', 'ideal_channel', 'br_amp', 'up_down'])

In [7]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.utils import resample
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

转化为xgb训练格式

In [8]:
haveno_X_train, haveno_X_test, haveno_y_train, haveno_y_test = train_test_split(have_no_df, have_no_label, test_size=0.2, random_state=42)
haveno_dtrain = xgb.DMatrix(haveno_X_train, label=haveno_y_train)
haveno_dtest = xgb.DMatrix(haveno_X_test, label=haveno_y_test)

alivedead_X_trian, alivedead_X_test, alivedead_y_train, alivedead_y_test = train_test_split(alive_dead_df, alive_dead_label, test_size=0.2,random_state=42)
alivedead_dtrain = xgb.DMatrix(alivedead_X_trian, label=alivedead_y_train)
alivedead_dtest = xgb.DMatrix(alivedead_X_test, label=alivedead_y_test)

In [9]:
def xgboost_train(params, dtrain, dtest, num_boost_round=300):
    evals = [(dtrain, 'trian'), (dtest, 'eval')]
    model = xgb.train(
        params,
        dtrain,
        num_boost_round=1000,  # 最大迭代次数
        evals=evals,
        verbose_eval=10,       # 每10轮打印一次日志
        early_stopping_rounds=10  # 提前停止
    )
    return model

In [10]:
#4. 设置模型参数二分类
params = {
    'eval_metric': 'logloss',        # 评估指标
    'eta': 0.015,                    # 学习率
    'max_depth': 6,                  # 树的最大深度
    'subsample': 0.8,                # 样本采样比例
    'colsample_bytree': 0.8,         # 特征采样比例
    'seed': 42,                      # 随机种子
}

训练模型

In [11]:
haveno_model = xgboost_train(params, haveno_dtrain, haveno_dtest)
alivedead_model = xgboost_train(params, alivedead_dtrain, alivedead_dtest)

[0]	trian-logloss:0.65266	eval-logloss:0.65343
[10]	trian-logloss:0.52744	eval-logloss:0.52829
[20]	trian-logloss:0.43370	eval-logloss:0.43462
[30]	trian-logloss:0.36003	eval-logloss:0.36100
[40]	trian-logloss:0.30094	eval-logloss:0.30194
[50]	trian-logloss:0.25318	eval-logloss:0.25425
[60]	trian-logloss:0.21372	eval-logloss:0.21486
[70]	trian-logloss:0.18100	eval-logloss:0.18218
[80]	trian-logloss:0.15409	eval-logloss:0.15531
[90]	trian-logloss:0.13110	eval-logloss:0.13237
[100]	trian-logloss:0.11219	eval-logloss:0.11352
[110]	trian-logloss:0.09617	eval-logloss:0.09756
[120]	trian-logloss:0.08245	eval-logloss:0.08387
[130]	trian-logloss:0.07075	eval-logloss:0.07221
[140]	trian-logloss:0.06079	eval-logloss:0.06229
[150]	trian-logloss:0.05232	eval-logloss:0.05386
[160]	trian-logloss:0.04512	eval-logloss:0.04671
[170]	trian-logloss:0.03906	eval-logloss:0.04069
[180]	trian-logloss:0.03368	eval-logloss:0.03534
[190]	trian-logloss:0.02907	eval-logloss:0.03077
[200]	trian-logloss:0.02519	eva

评估结果

In [12]:
haveno_y_pred_prob = haveno_model.predict(haveno_dtest)
alivedead_y_pred_prob = alivedead_model.predict(alivedead_dtest)

In [13]:
haveno_y_pred = [1 if x > 0.5 else 0 for x in haveno_y_pred_prob]
alivedead_y_pred = [1 if x > 0.5 else 0 for x in alivedead_y_pred_prob]

In [14]:
print('有猪无猪测试结果：')
print(f"准确率: {accuracy_score(haveno_y_test, haveno_y_pred):.4f}")
print(f"精确率: {precision_score(haveno_y_test, haveno_y_pred):.4f}")
print(f"召回率: {recall_score(haveno_y_test, haveno_y_pred):.4f}")
print(f"F1分数: {f1_score(haveno_y_test, haveno_y_pred):.4f}")
print(f"AUC分数: {roc_auc_score(haveno_y_test, haveno_y_pred_prob):.4f}")
print("混淆矩阵:\n", confusion_matrix(haveno_y_test, haveno_y_pred))

print('--------------------------------------------------')
print('活死猪测试结果：')
print(f"准确率: {accuracy_score(alivedead_y_test, alivedead_y_pred):.4f}")
print(f"精确率: {precision_score(alivedead_y_test, alivedead_y_pred):.4f}")
print(f"召回率: {recall_score(alivedead_y_test, alivedead_y_pred):.4f}")
print(f"F1分数: {f1_score(alivedead_y_test, alivedead_y_pred):.4f}")
print(f"AUC分数: {roc_auc_score(alivedead_y_test, alivedead_y_pred_prob):.4f}")
print("混淆矩阵:\n", confusion_matrix(alivedead_y_test, alivedead_y_pred))
# # 特征重要性分析
# importance = model.get_score(importance_type='weight')
# importance_df = pd.DataFrame({
#     'feature': list(importance.keys()),
#     'importance': list(importance.values())
# }).sort_values('importance', ascending=False)

# print("\n特征重要性(weight):")
# print(importance_df)

# importance = model.get_score(importance_type='gain')
# importance_df = pd.DataFrame({
#     'feature': list(importance.keys()),
#     'importance': list(importance.values())
# }).sort_values('importance', ascending=False)

# print("\n特征重要性(gain):")
# print(importance_df)

有猪无猪测试结果：
准确率: 0.9991
精确率: 0.9987
召回率: 0.9998
F1分数: 0.9993
AUC分数: 1.0000
混淆矩阵:
 [[4028    8]
 [   1 6341]]
--------------------------------------------------
活死猪测试结果：
准确率: 0.9937
精确率: 0.9927
召回率: 0.9990
F1分数: 0.9958
AUC分数: 0.9997
混淆矩阵:
 [[1540   35]
 [   5 4776]]


TODO:活死猪半监督训练过程

In [56]:
# 实现整个xgb模型半监督训练流程封装，包含初始训练、后续置信度不高数据训练、评估指标计算、保存模型
class SemiSupervisedTraining:
    def __init__(self, have_no_dataset, alive_dead_dataset, nolabel_dataset, have_no_label, alive_dead_label, logic_label,
                 xgb_params,
                 save_path,
                 confidence_threshold=0.9,
                 high_confidence_ratio=0.2,
                 max_iterations=5):
        '''
        xgb半监督学习流程
        have_no_dataset, alive_dead_dataset, nolabel_feature 三个数据集
        xgb_params: xgb训练参数
        save_path: 模型保存路径
        confidence_threshold: 半监督训练时置信度阈值
        high_confidence_ration: 半监督训练每次迭代中高置信度样本比例
        max_iterations: 最大迭代次数
        '''
        self.have_no_dataset = np.array(have_no_dataset)
        self.alive_dead_dataset = np.array(alive_dead_dataset)
        self.nolabel_dataset = np.array(nolabel_dataset)
        self.have_no_label = have_no_label
        self.alive_dead_label = alive_dead_label
        self.logic_label= logic_label
        self.xgb_params = xgb_params
        self.save_path = save_path
        self.confidence_threshold = confidence_threshold
        self.high_confidence_ratio = high_confidence_ratio
        self.max_iterations = max_iterations
        self.haveno_model = None
        self.alivedead_model = None
    
    def xgboost_train(self, train_data, eval_data, train_label, eval_label, num_boost_round=300):
        '''
        xgb训练代码
        '''
        dtrain = xgb.DMatrix(train_data, label = train_label)
        dtest = xgb.DMatrix(eval_data, label=eval_label)
        evals = [(dtrain, 'trian'), (dtest, 'eval')]
        model = xgb.train(
            params,
            dtrain,
            num_boost_round=1000,  # 最大迭代次数
            evals=evals,
            verbose_eval=10,       # 每10轮打印一次日志
            early_stopping_rounds=10  # 提前停止
        )
        return model
    
    def get_pred_prob(self, test_data, type):
        '''
        获得模型的输出概率
        '''
        dtest = xgb.DMatrix(test_data)
        pred_prob = None
        if type=='have_no':
            pred_prob = self.haveno_model.predict(dtest)
        elif type=='alive_dead':
            pred_prob = self.alivedead_model.predict(dtest)
        else:
            print('type有误！请修改为有无猪\'have_no\',活死猪\'alive_dead\'')
        return pred_prob
    
    def evaluate(self, test_data, test_label, type):
        # TODO:两个模型的评估，有无猪偏向有猪的精确率和召回率，活死猪偏向死猪的召回率和活猪的精确率
        test_pred_prob = self.get_pred_prob(test_data, type)
        test_pred  = [1 if x>0.5 else 0 for x in test_pred_prob]
        print(f'{type}测试结果：')
        print(f"准确率: {accuracy_score(haveno_y_test, haveno_y_pred):.4f}")
        print(f"精确率: {precision_score(haveno_y_test, haveno_y_pred):.4f}")
        print(f"召回率: {recall_score(haveno_y_test, haveno_y_pred):.4f}")
        print(f"F1分数: {f1_score(haveno_y_test, haveno_y_pred):.4f}")
        print(f"AUC分数: {roc_auc_score(haveno_y_test, haveno_y_pred_prob):.4f}")
        cm = confusion_matrix(test_label, test_pred)
        tn, fp, fn, tp = cm.ravel()
        print("混淆矩阵:")
        print('\tpred N\tpred P')
        print(f'label N\t{tn}\t{fp}')
        print(f'label P\t{fn}\t{tp}')
        if type  ==  'have_no':
            # 有无猪评估 计算有猪的精确率和召回率
            have_precision = tp/(fp+tp)
            have_recall = tp/(tp+fn)
            print(f'有猪精确率:{have_precision:.4f}')
            print(f'有猪召回率:{have_recall:.4f}')
        elif type == 'alive_dead':
            # 活死猪评估 计算死猪的召回率和活猪的精确率
            dead_recall = tn/(tn+fp)
            alive_precision = tp/(tp+fp)
            print(f'死猪召回率:{dead_recall:.4f}')
            print(f'活猪精确率:{alive_precision:.4f}')
        else:
            print('type有误！请修改为有无猪\'have_no\',活死猪\'alive_dead\'')
    
    def fit(self):
        '''
        半监督训练流程
        '''
        # 有标签数据训练得到basemodel
        have_no_df = pd.DataFrame(self.have_no_dataset, columns=['amp_max', 'amp_min', 'amp_mean', 'max_channel', 'ideal_channel', 'br_amp', 'up_down'])
        alive_dead_df = pd.DataFrame(self.alive_dead_dataset, columns=['amp_max', 'amp_min', 'amp_mean', 'max_channel', 'ideal_channel', 'br_amp', 'up_down'])
        nolabel_df = pd.DataFrame(self.nolabel_dataset, columns=['amp_max', 'amp_min', 'amp_mean', 'max_channel', 'ideal_channel', 'br_amp', 'up_down'])
        # 划分初始训练集、测试集、验证集 比例：0.7 0.15 0.15
        haveno_X_train, haveno_X_rest, haveno_y_train, haveno_y_rest = train_test_split(have_no_df, have_no_label, test_size=0.3, random_state=42)
        haveno_X_val, haveno_X_test, haveno_y_val, haveno_y_test = train_test_split(haveno_X_rest, haveno_y_rest, test_size=0.5, random_state=42)

        alivedead_X_trian, alivedead_X_rest, alivedead_y_train, alivedead_y_rest = train_test_split(alive_dead_df, alive_dead_label, test_size=0.3,random_state=42)
        alivedead_X_val, alivedead_X_test, alivedead_y_val, alivedead_y_test = train_test_split(alivedead_X_rest, alivedead_y_rest, test_size=0.5, random_state=42)

        self.haveno_model = self.xgboost_train(haveno_X_train, haveno_X_val, haveno_y_train, haveno_y_val)
        self.alivedead_model = self.xgboost_train(alivedead_X_trian, alivedead_X_val, alivedead_y_train, alivedead_y_val)
        # basemodel 评估
        print('初始模型评估：')
        self.evaluate(haveno_X_test, haveno_y_test, 'have_no')
        self.evaluate(alivedead_X_test, alivedead_y_test, 'alive_dead')

        logic_haveno_mask = np.full(len(self.logic_label), -1)
        logic_alivedead_mask = np.full(len(self.logic_label), -1)
        logic_haveno_label = np.array([0 if x==0 else 1 for x in self.logic_label])
        logic_alivedead_label = np.array([x-1 for x in self.logic_label]) # 将无猪标签置为-1过滤
        for iter in range(self.max_iterations):
            print(f'===========Iteration {iter+1}===========')
            # 先用当前模型对nolabel_dataset进行预测
            haveno_prob = self.get_pred_prob(nolabel_df, 'have_no')
            alivedead_prob = self.get_pred_prob(nolabel_df, 'alive_dead')
            haveno_confidence = np.array([x if x>0.5 else 1-x for x in haveno_prob])
            alivedead_confidence = np.array([x if x>0.5 else 1-x for x in alivedead_prob])
            haveno_pred = np.array([1 if x>0.5 else 0 for x in haveno_prob])
            alivedead_pred = np.array([1 if x>0.5 else 0 for x in alivedead_prob])
            # 将高置信度样本加入迭代训练
            unlabeled_haveno_mask = logic_haveno_mask == -1
            high_haveno_mask = (haveno_confidence > self.confidence_threshold) & \
                   (haveno_pred == logic_haveno_label) & \
                   unlabeled_haveno_mask
            
            unlabeled_alivedead_mask = logic_alivedead_mask == -1
            high_alivedead_mask = (alivedead_confidence > self.confidence_threshold) & \
                      (alivedead_pred == logic_alivedead_label) & \
                      unlabeled_alivedead_mask
            # 计算要添加的高置信度样本数量
            n_high_haveno = int(self.high_confidence_ratio * np.sum(unlabeled_haveno_mask))
            n_high_haveno = min(n_high_haveno, np.sum(high_haveno_mask))
            n_high_alivedead = int(self.high_confidence_ratio * np.sum(unlabeled_alivedead_mask))
            n_high_alivedead = min(n_high_alivedead, np.sum(high_alivedead_mask))

            haveno_flag = n_high_haveno == 0
            alivedead_flag = n_high_alivedead == 0
            if haveno_flag and alivedead_flag:
                print("没有高置信度样本，提前终止")
                break

            if not haveno_flag:
                high_conf_indices = np.where(high_haveno_mask)[0]
                selected_indices = resample(
                    high_conf_indices,
                    n_samples=n_high_haveno,
                    replace=False,
                    random_state=42
                )
                # 该次迭代添加的标签
                curiter_label = haveno_pred[selected_indices]
                # 该次迭代添加的数据
                curiter_data = self.nolabel_dataset[selected_indices]
                logic_haveno_mask[selected_indices] = 1
                curiter_df = pd.DataFrame(curiter_data, columns=['amp_max', 'amp_min', 'amp_mean', 'max_channel', 'ideal_channel', 'br_amp', 'up_down'])
                curiter_X_train, curiter_X_val, curiter_y_train, curiter_y_val = train_test_split(curiter_df, curiter_label, test_size=0.15, random_state=42)
                haveno_X_train = pd.concat([haveno_X_train, curiter_X_train], axis=0)
                haveno_y_train = np.concatenate((haveno_y_train, curiter_y_train))
                haveno_X_val = pd.concat([haveno_X_val, curiter_X_val], axis=0)
                haveno_y_val = np.concatenate((haveno_y_val, curiter_y_val))
                self.haveno_model = self.xgboost_train(haveno_X_train, haveno_X_val, haveno_y_train, haveno_y_val)
                self.evaluate(haveno_X_test, haveno_y_test, 'have_no')
            else:
                print("有无猪无新样本添加")
            
            if not alivedead_flag:
                high_conf_indices = np.where(high_alivedead_mask)[0]
                selected_indices = resample(
                    high_conf_indices,
                    n_samples=n_high_alivedead,
                    replace=False,
                    random_state=42
                )
                # 该次迭代添加的标签
                curiter_label = alivedead_pred[selected_indices]
                # 该次迭代添加的数据
                curiter_data = self.nolabel_dataset[selected_indices]
                logic_alivedead_mask[selected_indices] = 1
                curiter_df = pd.DataFrame(curiter_data, columns=['amp_max', 'amp_min', 'amp_mean', 'max_channel', 'ideal_channel', 'br_amp', 'up_down'])
                curiter_X_train, curiter_X_val, curiter_y_train, curiter_y_val = train_test_split(curiter_df, curiter_label, test_size=0.15, random_state=42)
                alivedead_X_trian = pd.concat([alivedead_X_trian, curiter_X_train], axis=0)
                alivedead_y_train = np.concatenate((alivedead_y_train, curiter_y_train))
                alivedead_X_val = pd.concat([alivedead_X_val, curiter_X_val], axis=0)
                alivedead_y_val = np.concatenate((alivedead_y_val, curiter_y_val))
                self.alivedead_model = self.xgboost_train(alivedead_X_trian, alivedead_X_val, alivedead_y_train, alivedead_y_val)
                self.evaluate(alivedead_X_test, alivedead_y_test, 'alive_dead')
            else:
                print("活死猪无新样本添加")

In [57]:
params = {
    'eval_metric': 'logloss',        # 评估指标
    'eta': 0.015,                    # 学习率
    'max_depth': 6,                  # 树的最大深度
    'subsample': 0.8,                # 样本采样比例
    'colsample_bytree': 0.8,         # 特征采样比例
    'seed': 42,                      # 随机种子
}

In [58]:
train = SemiSupervisedTraining(have_no_dataset, alive_dead_dataset, nolabel_dataset, have_no_label, alive_dead_label, logic_label, params, None)

In [59]:
train.fit()

[0]	trian-logloss:0.65350	eval-logloss:0.65388
[10]	trian-logloss:0.52822	eval-logloss:0.52872
[20]	trian-logloss:0.43431	eval-logloss:0.43495
[30]	trian-logloss:0.36054	eval-logloss:0.36120
[40]	trian-logloss:0.30137	eval-logloss:0.30204
[50]	trian-logloss:0.25355	eval-logloss:0.25425
[60]	trian-logloss:0.21401	eval-logloss:0.21475
[70]	trian-logloss:0.18125	eval-logloss:0.18197
[80]	trian-logloss:0.15431	eval-logloss:0.15504
[90]	trian-logloss:0.13130	eval-logloss:0.13201
[100]	trian-logloss:0.11234	eval-logloss:0.11310
[110]	trian-logloss:0.09631	eval-logloss:0.09707
[120]	trian-logloss:0.08257	eval-logloss:0.08334
[130]	trian-logloss:0.07086	eval-logloss:0.07163
[140]	trian-logloss:0.06088	eval-logloss:0.06165
[150]	trian-logloss:0.05239	eval-logloss:0.05317
[160]	trian-logloss:0.04519	eval-logloss:0.04597
[170]	trian-logloss:0.03911	eval-logloss:0.03990
[180]	trian-logloss:0.03373	eval-logloss:0.03450
[190]	trian-logloss:0.02912	eval-logloss:0.02989
[200]	trian-logloss:0.02524	eva